In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [28]:
def get_html(url = 'https://sindipetrosp.org.br/noticias/'):
    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [30]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%B %d, %Y")
                
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue


    return news_links

In [35]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    for link, date in news_links:
        if date < min_date:
            break
        else:
            validated_links.append([link, date])

    return validated_links



In [36]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')

    title = soup.find('h1').text
    div = soup.find('div', class_='penci-entry-content entry-content')

    paragraphs = div.find_all('p')
    return title, paragraphs


In [37]:
def main():
    url = 'https://sindipetrosp.org.br/noticias/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_news_links = get_validated_links(news_links)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'SP',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph.text,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [38]:
result = main()

df = pd.DataFrame(result)

df2 = df.explode('paragrafo')
df2.to_dict('records')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [01:09<00:00,  2.33s/it]


[{'sindicato': 'SP',
  'url': 'https://sindipetrosp.org.br/sindipetro-unificado-convoca-filiados-para-reinauguracao-da-sede-de-sao-paulo/',
  'titulo': 'Sindipetro Unificado convoca filiados para reinauguração da sede de São Paulo',
  'data': Timestamp('2025-08-18 00:00:00'),
  'paragrafo': 'Evento ocorre no dia 2 de setembro, conta com homenagem a fundadores do sindicato, palestra sobre assuntos da categoria e confraternização',
  'num_paragrafo': 1},
 {'sindicato': 'SP',
  'url': 'https://sindipetrosp.org.br/sindipetro-unificado-convoca-filiados-para-reinauguracao-da-sede-de-sao-paulo/',
  'titulo': 'Sindipetro Unificado convoca filiados para reinauguração da sede de São Paulo',
  'data': Timestamp('2025-08-18 00:00:00'),
  'paragrafo': 'Parcialmente fechada desde 2020, a sede de São Paulo do Sindipetro Unificado finalmente volta a funcionar. A partir de agora, a sede passa a atender presencialmente às terças-feiras, e excepcionalmente, em outros dias, mediante agendamento prévio. No